# CS590-04 (Deep Learning) - Final Project - EEG
## Setup

In [1]:
!pip install -q mne pyedflib torch torchvision torchaudio


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import numpy as np
import mne
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings(
    "ignore",
    message="Limited .* annotation.* outside the data range"
)
mne.set_log_level('WARNING')

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## Data Processing

In [ ]:
# Point to the local files folder (sibling of this notebook)
DATA_DIR = os.path.join(os.getcwd(), "files")

RUNS = ["R03", "R04", "R07", "R08", "R11", "R12"] #Only close either left or right hand. (Task 1)

SFREQ = 160
TMIN = 0.0
TMAX = 2.0   # 2-second windows

BATCH_SIZE = 64

In [4]:
def load_subject(subject_id):
    subject = f"S{subject_id:03d}"
    raws = []

    for run in RUNS:
        path = os.path.join(DATA_DIR, subject, f"{subject}{run}.edf")
        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
        raw.filter(1., 40., verbose=False)
        raws.append(raw)

    raw = mne.concatenate_raws(raws)
    events, event_id = mne.events_from_annotations(raw)
    event_dict = {'T1': event_id['T1'], 'T2': event_id['T2']}

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_dict,
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    y = epochs.events[:, -1]
    y = (y == event_id['T2']).astype(int)

    target_len = int(SFREQ * (TMAX - TMIN))  # 320

    X_fixed = []
    for x in X:
        if x.shape[1] >= target_len:
            x = x[:, :target_len]
        else:
            pad = target_len - x.shape[1]
            x = np.pad(x, ((0, 0), (0, pad)))
        X_fixed.append(x)

    X = np.stack(X_fixed)
    return X, y


def load_subjects(subjects):
    X_all, y_all = [], []
    for s in subjects:
        X, y = load_subject(s)
        X_all.append(X)
        y_all.append(y)
    return np.concatenate(X_all), np.concatenate(y_all)

In [5]:
subjects = list(range(1, 110))

rng = np.random.default_rng(seed=42)
rng.shuffle(subjects)

train_subjects = subjects[:80]
val_subjects   = subjects[80:90]
test_subjects  = subjects[90:]

print("Train:", train_subjects[:10], "...")
print("Val:",   val_subjects)
print("Test:",  test_subjects)

Train: [34, 19, 69, 29, 107, 1, 25, 79, 101, 89] ...
Val: [68, 35, 31, 59, 75, 97, 108, 55, 50, 7]
Test: [99, 12, 94, 42, 23, 20, 36, 65, 48, 13, 54, 15, 14, 67, 37, 98, 66, 78, 9]


In [6]:
def preprocess(X):
    X = (X - X.mean(axis=(1, 2), keepdims=True)) / (X.std(axis=(1, 2), keepdims=True) + 1e-6)
    X = X[:, np.newaxis, :, :]
    return X


class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Models

In [7]:
class EEGNet(nn.Module):
    def __init__(self, n_channels=64, n_samples=320, n_classes=2):
        super().__init__()

        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )
        self.depthwise = nn.Sequential(
            nn.Conv2d(8, 16, (n_channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.25)
        )
        self.separable = nn.Sequential(
            nn.Conv2d(16, 16, (1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(0.25)
        )
        self.classifier = nn.Linear(16 * (n_samples // 32), n_classes)

    def forward(self, x):
        x = self.firstconv(x)
        x = self.depthwise(x)
        x = self.separable(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

In [8]:
class EEG_CNN_3Layer(nn.Module):
    def __init__(self, n_channels=64, n_samples=320, n_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((1, 2)),

            nn.Conv2d(16, 32, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((1, 2)),

            nn.Conv2d(32, 64, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((1, 2))
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.features(dummy)
            self.flatten_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

In [13]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=(3, 5), padding=(1, 2), bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=(3, 5), padding=(1, 2), bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU()

        # 1x1 projection to match channels when they change
        self.skip = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.skip(x))


class EEG_ResNet(nn.Module):
    """
    ResNet-style model with the same 3-stage structure as EEG_CNN_3Layer.
    Each stage: ResidualBlock (conv->BN->ReLU->conv->BN + skip) + MaxPool(1,2).
    Channel progression: 1 -> 16 -> 32 -> 64, matching the baseline CNN.
    """
    def __init__(self, n_channels=64, n_samples=320, n_classes=2):
        super().__init__()

        # Lightweight stem to lift from 1 -> 16 channels
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, 5), padding=(1, 2), bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )

        self.stage1 = nn.Sequential(ResidualBlock(16, 16), nn.MaxPool2d((1, 2)))
        self.stage2 = nn.Sequential(ResidualBlock(16, 32), nn.MaxPool2d((1, 2)))
        self.stage3 = nn.Sequential(ResidualBlock(32, 64), nn.MaxPool2d((1, 2)))

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.stem(dummy)
            out = self.stage1(out)
            out = self.stage2(out)
            out = self.stage3(out)
            self.flatten_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

## Training + Validation Accuracy

In [14]:
def train_model(model, train_loader, val_loader, epochs=10):
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                preds = model(X).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += len(y)

        val_acc = correct / total
        print(f"Epoch {epoch+1} | Loss: {total_loss:.2f} | Val Acc: {val_acc:.3f}")

In [15]:
print("Loading data...")
X_train, y_train = load_subjects(train_subjects)
X_val,   y_val   = load_subjects(val_subjects)
X_test,  y_test  = load_subjects(test_subjects)

X_train = preprocess(X_train)
X_val   = preprocess(X_val)
X_test  = preprocess(X_test)

train_ds = EEGDataset(X_train, y_train)
val_ds   = EEGDataset(X_val,   y_val)
test_ds  = EEGDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

# model = EEGNet()
# model = EEG_CNN_3Layer()
model = EEG_ResNet()

train_model(model, train_loader, val_loader, epochs=10)

Loading data...
Epoch 1 | Loss: 272.62 | Val Acc: 0.728
Epoch 2 | Loss: 58.03 | Val Acc: 0.796
Epoch 3 | Loss: 52.38 | Val Acc: 0.792
Epoch 4 | Loss: 47.07 | Val Acc: 0.814
Epoch 5 | Loss: 44.29 | Val Acc: 0.824
Epoch 6 | Loss: 42.14 | Val Acc: 0.812
Epoch 7 | Loss: 39.83 | Val Acc: 0.809
Epoch 8 | Loss: 35.46 | Val Acc: 0.811
Epoch 9 | Loss: 33.93 | Val Acc: 0.821
Epoch 10 | Loss: 31.98 | Val Acc: 0.806


## Final Evaluation

In [16]:
def evaluate(model, loader):
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += len(y)
    return correct / total

In [17]:
test_acc = evaluate(model, test_loader)
print("Final Test Accuracy:", test_acc)

Final Test Accuracy: 0.7672514619883041
